## TreeQSM data 

**Gabon** 4 plots

Work with QSMs from TreeQSM (.mat files) to extract tree-level metrics.

In [ ]:
# Library for MATLAB files
%pip install pymatreader requests

from scipy.io import loadmat

import pandas as pd
from pymatreader import read_mat

import math
from pathlib import Path

import os

## LOOP all folders ##

In [ ]:
allfolders_path = Path('/home/jovyan/Examples_and_Data/Workshops/BIOMASS-Coding2026/TLS/Lope/')

subfolders = [f for f in os.listdir(allfolders_path) if os.path.isdir(os.path.join(allfolders_path, f))]

print(subfolders)

In [ ]:
# Filter for directories AND ensure '-ply' is not in the name
subfolders = [
    f for f in os.listdir(allfolders_path) 
    if os.path.isdir(os.path.join(allfolders_path, f)) and '-ply' not in f
]

print(subfolders)

In [ ]:
#allfolders_path = Path('/home/jovyan/Examples_and_Data/Workshops/BIOMASS-Coding2026/TLS/Lope/')

# 1. Select all subfolders that do not have '-ply' in their names
subfolders = [
    f for f in allfolders_path.iterdir() 
    if f.is_dir() and '-ply' not in f.name
]

# 2. Outer Loop: Process each subfolder independently
for folder_path in subfolders:
    print(f"\n==================================================================")
    print(f" PROCESSING POPULATION: {folder_path.name}")
    print(f"==================================================================")
    
    # List to hold data for each tree observation within THIS specific subfolder
    observations = []

    # 3. Inner Loop: Loop through all .mat files in this specific directory
    for file_path in folder_path.glob('*.mat'):
        try:
            # Load the file
            data = read_mat(file_path)

            # Access the main QSM structure
            if "qsm" in data:
                qsm = data["qsm"]
                treedata = qsm["treedata"]

                # Safely extract Volume (checking both standard TreeQSM naming variations)
                raw_volume = treedata.get("TotalVolume", treedata.get("volume", None))

                # Safely extract Height
                height = treedata.get("TreeHeight", treedata.get("height", None))

                # Safely extract DBH variations
                dbh_qsm = treedata.get("DBHqsm", None)
                dbh_cyl = treedata.get("DBHcyl", None)

                # Process volume if it exists
                if raw_volume is not None:
                    volume_m3 = raw_volume / 1000
                else:
                    volume_m3 = None

                # Append the data row to our collection
                observations.append(
                    {
                        "File_Name": file_path.name,
                        "Total_Volume_m3": volume_m3,
                        "Tree_Height_m": (float(height) if height is not None else None),
                        "DBHqsm": (float(dbh_qsm) if dbh_qsm is not None else None),
                        "DBHcyl": (float(dbh_cyl) if dbh_cyl is not None else None),
                    }
                )

        except Exception as e:
            print(f"Error processing file {file_path.name}: {e}")

    # If no .mat files were found or successfully parsed, skip to the next folder
    if not observations:
        print(f"No valid tree observations found in {folder_path.name}. Skipping calculations.")
        continue

    # 4. Compile all observations for this population into a Pandas DataFrame
    df = pd.DataFrame(observations)

    # Display the resulting DataFrame table for this folder
    print(f"\n--- Processed Tree Observations ({folder_path.name}) ---")
    print(df)

    # 5. Calculate Totals
    total_volume = df["Total_Volume_m3"].sum()
    print(f"\nGrand Total Volume across all trees: {total_volume:.2f} m³\n")

    # 6. Define your plot size in hectares to get the per-hectare rate
    plot_area_ha = 1.0
    total_volume_per_ha = total_volume / plot_area_ha

    # 7. Multiply the total volume per hectare directly by the thresholds
    biomass_low = total_volume_per_ha * 0.4 
    biomass_med = total_volume_per_ha * 0.6 
    biomass_high = total_volume_per_ha * 0.8 

    # 8. Print the direct range
    print("--- Biomass Range Per Hectare ---")
    print(f"Low (0.4):    {biomass_low:.2f} Mg/ha")
    print(f"Medium (0.6): {biomass_med:.2f} Mg/ha")
    print(f"High (0.8):   {biomass_high:.2f} Mg/ha \n")

    # 9. Generate Summary Statistics (Now including DBHqsm and DBHcyl columns)
    summary_stats = df[["Total_Volume_m3", "Tree_Height_m", "DBHqsm", "DBHcyl"]].describe()

    # 10. Add Custom Row for "Sum" bundled in the same table (Only volume is totaled)
    summary_stats.loc["sum"] = [df["Total_Volume_m3"].sum(), None, None, None]

    print("--- Summary Statistics Table ---")
    print(summary_stats.round(4))

    # Optional: Export reports dynamically inside each specific subfolder
    # df.to_csv(folder_path / f'{folder_path.name}_tree_summary.csv', index=False)
    # summary_stats.to_csv(folder_path / f'{folder_path.name}_tree_summary_statistics.csv')
    
    # Export DF
    df.to_csv(f'/home/jovyan/{folder_path.name}_tree_summary.csv', index=False)
    summary_stats.to_csv(f'/home/jovyan/{folder_path.name}_statistics.csv')



In [ ]:
df

**The End** ________

## Summary Statistics ###

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# 1. Define the sites and the base directory where your CSV files are saved
base_path = Path('/home/jovyan/')
sites = ['LPG-01', 'OKO-01', 'OKO-02', 'OKO-03']

# 2. Loop through each site to open its CSV and plot the histograms
for site in sites:
    # Construct the file path matching your naming convention
    #csv_file = base_path / f'{site}_statistics.csv'
    csv_file = base_path / f'{site}_tree_summary.csv'
    
    # Check if the file exists before attempting to read it
    if not csv_file.exists():
        print(f"File not found: {csv_file}")
        continue
        
    print(f"Processing and plotting data for site: {site}")
    
    # 3. Load the data frame
    df = pd.read_csv(csv_file)
    
    # 4. Check if we accidentally loaded the summary stats instead of the raw data.
    # The raw data file contains a 'File_Name' column. 
    # If the file has an unnamed index or row labels like 'mean', 'std', '50%', it's the wrong file!
    if 'File_Name' not in df.columns and df.iloc[:, 0].isin(['count', 'mean', 'std']).any():
        print(f"⚠️ Error: '{csv_file.name}' appears to contain aggregated summary statistics.")
        print(f"To plot histograms, please ensure this file contains the raw tree rows (with 'File_Name').\n")
        continue

    # 5. Convert DBHqsm from meters to centimeters safely
    if 'DBHqsm' in df.columns:
        df['DBHqsm_cm'] = df['DBHqsm'] * 100
    else:
        df['DBHqsm_cm'] = None

    # 6. Initialize a clean 1x3 panel figure side-by-side
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(f'Distributions for Site: {site}', fontsize=16, fontweight='bold')

    # Variable 1: Total Volume (m³)
    if 'Total_Volume_m3' in df.columns and df['Total_Volume_m3'].notna().any():
        axes[0].hist(df['Total_Volume_m3'].dropna(), bins=12, color='forestgreen', edgecolor='black', alpha=0.7)
        axes[0].set_title('Total Volume')
        axes[0].set_xlabel('Volume (m³)')
        axes[0].set_ylabel('Frequency')
        axes[0].set_xlim(0, 60)  # <-- Fixed X-axis limit
        axes[0].grid(axis='y', linestyle='--', alpha=0.5)
    else:
        axes[0].text(0.5, 0.5, 'No Data Available', ha='center', va='center')

    # Variable 2: Tree Height (m)
    if 'Tree_Height_m' in df.columns and df['Tree_Height_m'].notna().any():
        axes[1].hist(df['Tree_Height_m'].dropna(), bins=12, color='teal', edgecolor='black', alpha=0.7)
        axes[1].set_title('Tree Height')
        axes[1].set_xlabel('Height (m)')
        axes[1].set_ylabel('Frequency')
        axes[1].set_xlim(0, 50)  # <-- Fixed X-axis limit
        axes[1].grid(axis='y', linestyle='--', alpha=0.5)
    else:
        axes[1].text(0.5, 0.5, 'No Data Available', ha='center', va='center')

    # Variable 3: DBHqsm (cm)
    if 'DBHqsm_cm' in df.columns and df['DBHqsm_cm'].notna().any():
        axes[2].hist(df['DBHqsm_cm'].dropna(), bins=12, color='darkorange', edgecolor='black', alpha=0.7)
        axes[2].set_title('DBHqsm')
        axes[2].set_xlabel('DBH (cm)')
        axes[2].set_ylabel('Frequency')
        axes[2].set_xlim(0, 300)          # <-- Fixed X-axis limit
        axes[2].grid(axis='y', linestyle='--', alpha=0.5)
    else:
        axes[2].text(0.5, 0.5, 'No Data Available', ha='center', va='center')

    # 7. Adjust layouts and display
    plt.tight_layout()
    plt.show()


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# 1. Define the sites and the base directory where your CSV files are saved
base_path = Path('/home/jovyan/')
sites = ['LPG-01', 'OKO-01', 'OKO-02', 'OKO-03']

# 2. Loop through each site to open its CSV and plot the histograms
for site in sites:
    # Construct the file path matching your naming convention
    csv_file = base_path / f'{site}_tree_summary.csv'
    
    # Check if the file exists before attempting to read it
    if not csv_file.exists():
        print(f"File not found: {csv_file}")
        continue
        
    print(f"Processing and plotting data for site: {site}")
    
    # 3. Load the data frame
    df = pd.read_csv(csv_file)
    
    # 4. Check if we accidentally loaded the summary stats instead of the raw data.
    if 'File_Name' not in df.columns and df.iloc[:, 0].isin(['count', 'mean', 'std']).any():
        print(f"⚠️ Error: '{csv_file.name}' appears to contain aggregated summary statistics.")
        print(f"To plot histograms, please ensure this file contains the raw tree rows (with 'File_Name').\n")
        continue

    # 5. Convert DBHqsm from meters to centimeters safely
    if 'DBHqsm' in df.columns:
        df['DBHqsm_cm'] = df['DBHqsm'] * 100
    else:
        df['DBHqsm_cm'] = None

    # 6. Initialize a clean 1x3 panel figure side-by-side
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(f'Distributions for Site: {site}', fontsize=16, fontweight='bold')

    # Variable 1: Total Volume (m³)
    if 'Total_Volume_m3' in df.columns and df['Total_Volume_m3'].notna().any():
        valid_data = df['Total_Volume_m3'].dropna()
        axes[0].hist(valid_data, bins=12, color='forestgreen', edgecolor='black', alpha=0.7)
        axes[0].set_title('Total Volume')
        axes[0].set_xlabel('Volume (m³)')
        axes[0].set_ylabel('Frequency')
        axes[0].set_xlim(0, 60)
        axes[0].grid(axis='y', linestyle='--', alpha=0.5)
        
        # Calculate and plot 75th percentile line
        p75 = valid_data.quantile(0.75)
        axes[0].axvline(p75, color='red', linestyle='--', linewidth=1.5, label=f'75th% ({p75:.2f})')
        axes[0].legend()
    else:
        axes[0].text(0.5, 0.5, 'No Data Available', ha='center', va='center')

    # Variable 2: Tree Height (m)
    if 'Tree_Height_m' in df.columns and df['Tree_Height_m'].notna().any():
        valid_data = df['Tree_Height_m'].dropna()
        axes[1].hist(valid_data, bins=12, color='teal', edgecolor='black', alpha=0.7)
        axes[1].set_title('Tree Height')
        axes[1].set_xlabel('Height (m)')
        axes[1].set_ylabel('Frequency')
        axes[1].set_xlim(0, 50)
        axes[1].grid(axis='y', linestyle='--', alpha=0.5)
        
        # Calculate and plot 75th percentile line
        p75 = valid_data.quantile(0.75)
        axes[1].axvline(p75, color='red', linestyle='--', linewidth=1.5, label=f'75th% ({p75:.1f})')
        axes[1].legend()
    else:
        axes[1].text(0.5, 0.5, 'No Data Available', ha='center', va='center')

    # Variable 3: DBHqsm (cm)
    if 'DBHqsm_cm' in df.columns and df['DBHqsm_cm'].notna().any():
        valid_data = df['DBHqsm_cm'].dropna()
        axes[2].hist(valid_data, bins=12, color='darkorange', edgecolor='black', alpha=0.7)
        axes[2].set_title('DBHqsm')
        axes[2].set_xlabel('DBH (cm)')
        axes[2].set_ylabel('Frequency')
        axes[2].set_xlim(0, 300)
        axes[2].grid(axis='y', linestyle='--', alpha=0.5)
        
        # Calculate and plot 75th percentile line
        p75 = valid_data.quantile(0.75)
        axes[2].axvline(p75, color='red', linestyle='--', linewidth=1.5, label=f'75th% ({p75:.1f})')
        axes[2].legend()
    else:
        axes[2].text(0.5, 0.5, 'No Data Available', ha='center', va='center')

    # 7. Adjust layouts and display
    plt.tight_layout()
    plt.show()


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# 1. Define the sites and the base directory where your CSV files are saved
base_path = Path('/home/jovyan/')
sites = ['LPG-01', 'OKO-01', 'OKO-02', 'OKO-03']

# Map the single-pixel forest canopy height value for each site (OKO-02 uses the 1st row only)
canopy_heights = {
    'LPG-01': 34.696499,
    'OKO-01': 35.966438,
    'OKO-02': 41.091602,  # Second row dropped
    'OKO-03': 39.476715
}

# 2. Loop through each site to open its CSV and plot the histograms
for site in sites:
    # Construct the file path matching your naming convention
    csv_file = base_path / f'{site}_tree_summary.csv'
    
    # Check if the file exists before attempting to read it
    if not csv_file.exists():
        print(f"File not found: {csv_file}")
        continue
        
    print(f"Processing and plotting data for site: {site}")
    
    # 3. Load the data frame
    df = pd.read_csv(csv_file)
    
    # 4. Check if we accidentally loaded the summary stats instead of the raw data.
    if 'File_Name' not in df.columns and df.iloc[:, 0].isin(['count', 'mean', 'std']).any():
        print(f"⚠️ Error: '{csv_file.name}' appears to contain aggregated summary statistics.")
        print(f"To plot histograms, please ensure this file contains the raw tree rows (with 'File_Name').\n")
        continue

    # 5. Convert DBHqsm from meters to centimeters safely
    if 'DBHqsm' in df.columns:
        df['DBHqsm_cm'] = df['DBHqsm'] * 100
    else:
        df['DBHqsm_cm'] = None

    # 6. Initialize a clean 1x3 panel figure side-by-side
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(f'Distributions for Site: {site}', fontsize=16, fontweight='bold')

    # Variable 1: Total Volume (m³)
    if 'Total_Volume_m3' in df.columns and df['Total_Volume_m3'].notna().any():
        valid_data = df['Total_Volume_m3'].dropna()
        axes[0].hist(valid_data, bins=12, color='forestgreen', edgecolor='black', alpha=0.7)
        axes[0].set_title('Total Volume')
        axes[0].set_xlabel('Volume (m³)')
        axes[0].set_ylabel('Frequency')
        axes[0].set_xlim(0, 60)
        axes[0].grid(axis='y', linestyle='--', alpha=0.5)
        
        # Calculate and plot 75th percentile line
        p75 = valid_data.quantile(0.75)
        axes[0].axvline(p75, color='red', linestyle='--', linewidth=1.5, label=f'75th% ({p75:.2f})')
        axes[0].legend()
    else:
        axes[0].text(0.5, 0.5, 'No Data Available', ha='center', va='center')

    # Variable 2: Tree Height (m)
    if 'Tree_Height_m' in df.columns and df['Tree_Height_m'].notna().any():
        valid_data = df['Tree_Height_m'].dropna()
        axes[1].hist(valid_data, bins=12, color='teal', edgecolor='black', alpha=0.7)
        axes[1].set_title('Tree Height')
        axes[1].set_xlabel('Height (m)')
        axes[1].set_ylabel('Frequency')
        axes[1].set_xlim(0, 50)
        axes[1].grid(axis='y', linestyle='--', alpha=0.5)
        
        # Calculate and plot 75th percentile line
        p75 = valid_data.quantile(0.75)
        axes[1].axvline(p75, color='red', linestyle='--', linewidth=1.5, label=f'75th% ({p75:.1f})')
        
        # Overlay the single-pixel forest height canopy value
        site_canopy = canopy_heights[site]
        axes[1].axvline(site_canopy, color='royalblue', linestyle='-.', linewidth=2, label=f'Canopy H. ({site_canopy:.1f}m)')
        
        axes[1].legend()
    else:
        axes[1].text(0.5, 0.5, 'No Data Available', ha='center', va='center')

    # Variable 3: DBHqsm (cm)
    if 'DBHqsm_cm' in df.columns and df['DBHqsm_cm'].notna().any():
        valid_data = df['DBHqsm_cm'].dropna()
        axes[2].hist(valid_data, bins=12, color='darkorange', edgecolor='black', alpha=0.7)
        axes[2].set_title('DBHqsm')
        axes[2].set_xlabel('DBH (cm)')
        axes[2].set_ylabel('Frequency')
        axes[2].set_xlim(0, 300)
        axes[2].grid(axis='y', linestyle='--', alpha=0.5)
        
        # Calculate and plot 75th percentile line
        p75 = valid_data.quantile(0.75)
        axes[2].axvline(p75, color='red', linestyle='--', linewidth=1.5, label=f'75th% ({p75:.1f})')
        axes[2].legend()
    else:
        axes[2].text(0.5, 0.5, 'No Data Available', ha='center', va='center')

    # 7. Adjust layouts and display
    plt.tight_layout()
    plt.show()


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# 1. Define the sites and the base directory where your CSV files are saved
base_path = Path('/home/jovyan/')
sites = ['LPG-01', 'OKO-01', 'OKO-02', 'OKO-03']

# Map the single-pixel forest canopy height value for each site (OKO-02 uses the 1st row only)
canopy_heights = {
    'LPG-01': 34.696499,
    'OKO-01': 35.966438,
    'OKO-02': 41.091602,  # Second row dropped
    'OKO-03': 39.476715
}

# 2. Loop through each site to open its CSV and plot the histograms
for site in sites:
    # Construct the file path matching your naming convention
    csv_file = base_path / f'{site}_tree_summary.csv'
    
    # Check if the file exists before attempting to read it
    if not csv_file.exists():
        print(f"File not found: {csv_file}")
        continue
        
    print(f"Processing and plotting data for site: {site}")
    
    # 3. Load the data frame
    df = pd.read_csv(csv_file)
    
    # 4. Check if we accidentally loaded the summary stats instead of the raw data.
    if 'File_Name' not in df.columns and df.iloc[:, 0].isin(['count', 'mean', 'std']).any():
        print(f"⚠️ Error: '{csv_file.name}' appears to contain aggregated summary statistics.")
        print(f"To plot histograms, please ensure this file contains the raw tree rows (with 'File_Name').\n")
        continue

    # 5. Convert DBHqsm from meters to centimeters safely
    if 'DBHqsm' in df.columns:
        df['DBHqsm_cm'] = df['DBHqsm'] * 100
    else:
        df['DBHqsm_cm'] = None

    # 6. Initialize a clean 1x3 panel figure side-by-side
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(f'Distributions for Site: {site}', fontsize=16, fontweight='bold')

    # Variable 1: Total Volume (m³)
    if 'Total_Volume_m3' in df.columns and df['Total_Volume_m3'].notna().any():
        valid_data = df['Total_Volume_m3'].dropna()
        
        # Calculate grand total volume for this site
        total_site_volume = valid_data.sum()
        
        # Plot histogram
        axes[0].hist(valid_data, bins=12, color='forestgreen', edgecolor='black', alpha=0.7)
        axes[0].set_title('Total Volume')
        axes[0].set_xlabel('Volume (m³)')
        axes[0].set_ylabel('Frequency')
        axes[0].set_xlim(0, 60)
        axes[0].grid(axis='y', linestyle='--', alpha=0.5)
        
        # Calculate and plot 75th percentile line
        p75 = valid_data.quantile(0.75)
        axes[0].axvline(p75, color='red', linestyle='--', linewidth=1.5, label=f'75th% ({p75:.2f})')
        
        # Add a custom invisible proxy artist to display the Grand Total in the legend
        axes[0].plot([], [], ' ', label=f'Total Vol: {total_site_volume:.2f} m³')
        
        axes[0].legend()
    else:
        axes[0].text(0.5, 0.5, 'No Data Available', ha='center', va='center')

    # Variable 2: Tree Height (m)
    if 'Tree_Height_m' in df.columns and df['Tree_Height_m'].notna().any():
        valid_data = df['Tree_Height_m'].dropna()
        axes[1].hist(valid_data, bins=12, color='teal', edgecolor='black', alpha=0.7)
        axes[1].set_title('Tree Height')
        axes[1].set_xlabel('Height (m)')
        axes[1].set_ylabel('Frequency')
        axes[1].set_xlim(0, 50)
        axes[1].grid(axis='y', linestyle='--', alpha=0.5)
        
        # Calculate and plot 75th percentile line
        p75 = valid_data.quantile(0.75)
        axes[1].axvline(p75, color='red', linestyle='--', linewidth=1.5, label=f'75th% ({p75:.1f})')
        
        # Overlay the single-pixel forest height canopy value
        site_canopy = canopy_heights[site]
        axes[1].axvline(site_canopy, color='royalblue', linestyle='-.', linewidth=2, label=f'Canopy H. ({site_canopy:.1f}m)')
        
        axes[1].legend()
    else:
        axes[1].text(0.5, 0.5, 'No Data Available', ha='center', va='center')

    # Variable 3: DBHqsm (cm)
    if 'DBHqsm_cm' in df.columns and df['DBHqsm_cm'].notna().any():
        valid_data = df['DBHqsm_cm'].dropna()
        axes[2].hist(valid_data, bins=12, color='darkorange', edgecolor='black', alpha=0.7)
        axes[2].set_title('DBHqsm')
        axes[2].set_xlabel('DBH (cm)')
        axes[2].set_ylabel('Frequency')
        axes[2].set_xlim(0, 300)
        axes[2].grid(axis='y', linestyle='--', alpha=0.5)
        
        # Calculate and plot 75th percentile line
        p75 = valid_data.quantile(0.75)
        axes[2].axvline(p75, color='red', linestyle='--', linewidth=1.5, label=f'75th% ({p75:.1f})')
        axes[2].legend()
    else:
        axes[2].text(0.5, 0.5, 'No Data Available', ha='center', va='center')

    # 7. Adjust layouts and display
    plt.tight_layout()
    plt.show()


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# 1. Define the sites and the base directory where your CSV files are saved
base_path = Path('/home/jovyan/')
sites = ['LPG-01', 'OKO-01', 'OKO-02', 'OKO-03']

# Map the single-pixel forest canopy height value for each site (OKO-02 uses the 1st row only)
canopy_heights = {
    'LPG-01': 34.696499,
    'OKO-01': 35.966438,
    'OKO-02': 41.091602,  # Second row dropped
    'OKO-03': 39.476715
}

# 2. Loop through each site to open its CSV, plot, and export
for site in sites:
    # Construct the file path matching your naming convention
    csv_file = base_path / f'{site}_tree_summary.csv'
    
    # Check if the file exists before attempting to read it
    if not csv_file.exists():
        print(f"File not found: {csv_file}")
        continue
        
    print(f"Processing, plotting, and exporting data for site: {site}")
    
    # 3. Load the data frame
    df = pd.read_csv(csv_file)
    
    # 4. Check if we accidentally loaded the summary stats instead of the raw data.
    if 'File_Name' not in df.columns and df.iloc[:, 0].isin(['count', 'mean', 'std']).any():
        print(f"⚠️ Error: '{csv_file.name}' appears to contain aggregated summary statistics.")
        print(f"To plot histograms, please ensure this file contains the raw tree rows (with 'File_Name').\n")
        continue

    # 5. Convert DBHqsm from meters to centimeters safely
    if 'DBHqsm' in df.columns:
        df['DBHqsm_cm'] = df['DBHqsm'] * 100
    else:
        df['DBHqsm_cm'] = None

    # 6. Initialize a clean 1x3 panel figure side-by-side
    fig, axes = plt.subplots(1, 3, figsize=(20, 6))  # Slightly wider canvas for high-res layouts
    fig.suptitle(f'Distributions for Site: {site}', fontsize=18, fontweight='bold', y=1.02)

    # Variable 1: Total Volume (m³)
    if 'Total_Volume_m3' in df.columns and df['Total_Volume_m3'].notna().any():
        valid_data = df['Total_Volume_m3'].dropna()
        
        # Calculate grand total volume for this site
        total_site_volume = valid_data.sum()
        
        # Plot histogram
        axes[0].hist(valid_data, bins=12, color='forestgreen', edgecolor='black', alpha=0.7)
        axes[0].set_title('Total Volume', fontsize=14, fontweight='bold')
        axes[0].set_xlabel('Volume (m³)', fontsize=12)
        axes[0].set_ylabel('Frequency', fontsize=12)
        axes[0].set_xlim(0, 60)
        axes[0].grid(axis='y', linestyle='--', alpha=0.5)
        
        # Calculate and plot 75th percentile line
        p75 = valid_data.quantile(0.75)
        axes[0].axvline(p75, color='red', linestyle='--', linewidth=1.5, label=f'75th% ({p75:.2f})')
        
        # Add a custom invisible proxy artist to display the Grand Total with an enhanced larger font
        axes[0].plot([], [], ' ', label=f'Total Vol: {total_site_volume:.2f} m³')
        
        # Customize the legend to emphasize the Total Volume label text size
        leg = axes[0].legend(fontsize=13, loc='upper right', framealpha=0.9)
        # Iterate over labels to specifically target and bold/enlarge the Total Volume entry
        for text in leg.get_texts():
            if "Total Vol" in text.get_text():
                text.set_fontsize(14)
                text.set_weight('bold')
                text.set_color('darkgreen')
    else:
        axes[0].text(0.5, 0.5, 'No Data Available', ha='center', va='center')

    # Variable 2: Tree Height (m)
    if 'Tree_Height_m' in df.columns and df['Tree_Height_m'].notna().any():
        valid_data = df['Tree_Height_m'].dropna()
        axes[1].hist(valid_data, bins=12, color='teal', edgecolor='black', alpha=0.7)
        axes[1].set_title('Tree Height', fontsize=14, fontweight='bold')
        axes[1].set_xlabel('Height (m)', fontsize=12)
        axes[1].set_ylabel('Frequency', fontsize=12)
        axes[1].set_xlim(0, 50)
        axes[1].grid(axis='y', linestyle='--', alpha=0.5)
        
        # Calculate and plot 75th percentile line
        p75 = valid_data.quantile(0.75)
        axes[1].axvline(p75, color='red', linestyle='--', linewidth=1.5, label=f'75th% ({p75:.1f})')
        
        # Overlay the single-pixel forest height canopy value
        site_canopy = canopy_heights[site]
        axes[1].axvline(site_canopy, color='royalblue', linestyle='-.', linewidth=2, label=f'Canopy H. ({site_canopy:.1f}m)')
        
        axes[1].legend(fontsize=11)
    else:
        axes[1].text(0.5, 0.5, 'No Data Available', ha='center', va='center')

    # Variable 3: DBHqsm (cm)
    if 'DBHqsm_cm' in df.columns and df['DBHqsm_cm'].notna().any():
        valid_data = df['DBHqsm_cm'].dropna()
        axes[2].hist(valid_data, bins=12, color='darkorange', edgecolor='black', alpha=0.7)
        axes[2].set_title('DBHqsm', fontsize=14, fontweight='bold')
        axes[2].set_xlabel('DBH (cm)', fontsize=12)
        axes[2].set_ylabel('Frequency', fontsize=12)
        axes[2].set_xlim(0, 300)
        axes[2].grid(axis='y', linestyle='--', alpha=0.5)
        
        # Calculate and plot 75th percentile line
        p75 = valid_data.quantile(0.75)
        axes[2].axvline(p75, color='red', linestyle='--', linewidth=1.5, label=f'75th% ({p75:.1f})')
        axes[2].legend(fontsize=11)
    else:
        axes[2].text(0.5, 0.5, 'No Data Available', ha='center', va='center')

    # 7. Layout optimization and high-quality file export
    plt.tight_layout()
    
    # Save target file name string pattern mapping
    output_filename = base_path / f'{site}_distribution_plots_highres.png'
    
    # export with crisp, presentation-level 300 DPI layout settings
    plt.savefig(output_filename, dpi=300, bbox_inches='tight')
    print(f"Successfully saved high-resolution graphic to: {output_filename}")
    
    plt.show()
